# 1. How we got our results

This notebook gives the code for the data and the portfolio results. Run it in order. The supplied data files contain the dated stock scores and risk inputs. The code below also shows how to make these files from the source data. The source stock and CPI files need a data license. The supplied dated files let you run the portfolio test without access to those source files.

**Terms.** An alpha is a stock score that can help select a stock. A signal date is the date when a score is known. A holding return is the stock return after the next trade. A risk model gives a forecast of portfolio risk.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

# Open this notebook from the submission folder.
DATA = Path('dataset')
assert (DATA / 'analysis_panel.parquet').is_file()
plt.rcParams.update({'figure.figsize': (11, 5), 'axes.spines.top': False,
                     'axes.spines.right': False})


## 1.1 Make the DMD stock score

Dynamic mode decomposition (DMD) uses recent stock prices to find patterns. The code makes a price direction forecast for each stock. It selects a price window from results that are known on the signal date. The forecast is for the close after the trade to the next close. This date order prevents the use of future results.

In [2]:
import numpy as np

def dmd_predicts_up(
    prices: np.ndarray, horizon: int = 1, rank: int = 25, start_horizon: int = 0,
) -> bool:
    """Forecast direction between two future offsets from the last close."""
    prices = np.asarray(prices, dtype=float)
    n = len(prices)
    if (n < 4 or start_horizon < 0 or horizon <= start_horizon
            or not np.isfinite(prices).all() or (prices <= 0).any()):
        return False
    # Use the dated price values. This is the model input.
    logp = prices.copy()
    row_len = n // 2 + 1
    hankel = np.column_stack([logp[i:i + row_len] for i in range(n - row_len + 1)])
    x, y = hankel[:, :-1], hankel[:, 1:]
    try:
        u, singular, vh = np.linalg.svd(x, full_matrices=False)
        r = min(rank, len(singular))
        if r == 0 or singular[r - 1] <= np.finfo(float).eps:
            return False
        ur, vr = u[:, :r], vh[:r, :].T
        inv_s = np.diag(1.0 / singular[:r])
        reduced = ur.T @ y @ vr @ inv_s
        eigenvalues, modes = np.linalg.eig(reduced)
        good = np.isfinite(eigenvalues) & (np.abs(eigenvalues) > 1e-12)
        if not good.any():
            return False
        eigenvalues, modes = eigenvalues[good], modes[:, good]
        phi = y @ vr @ inv_s @ modes
        amplitudes = np.linalg.lstsq(phi, hankel, rcond=None)[0][:, 0]
        last = phi[0] @ (amplitudes * eigenvalues ** (hankel.shape[1] - 1 + start_horizon))
        future = phi[0] @ (amplitudes * eigenvalues ** (hankel.shape[1] - 1 + horizon))
        return bool(np.isfinite(last) and np.isfinite(future) and future > last)
    except (np.linalg.LinAlgError, FloatingPointError):
        return False


def walk_forward_stock_signal(
    prices: np.ndarray,
    *,
    windows: tuple[int, ...] = tuple(range(3, 25)),
    selection_lookback: int = 60,
    invert: bool = True,
) -> np.ndarray:
    """Daily online hit-rate selection for each stock.

    The forecast covers the close after execution (t+1) to the next close
    (t+2). Window selection uses only those outcomes known by signal date t.
    Output is 0 until enough history exists, otherwise
    +1/-1; the model's ``INVERT_SIGNALS=True`` convention is retained.
    """
    p = np.asarray(prices, dtype=float)
    t_count = len(p)
    pred = np.full((len(windows), t_count), np.nan)
    for wi, size in enumerate(windows):
        for t in range(size - 1, t_count):
            segment = p[t-size+1:t+1]
            if np.isfinite(segment).all():
                pred[wi, t] = 1.0 if dmd_predicts_up(
                    segment, horizon=2, start_horizon=1,
                ) else -1.0
    out = np.zeros(t_count, dtype=np.float32)
    for t in range(t_count):
        # At date t, only the result of a forecast made at t-2 is known.
        hi = t - 2
        lo = hi - selection_lookback + 1
        if lo < 0 or hi < lo:
            continue
        future = p[2:t+1] / p[1:t] - 1.0
        actual = np.sign(future)
        actual[~np.isfinite(future)] = 0
        scores = []
        for wi in range(len(windows)):
            hist = pred[wi, lo:hi+1]
            truth = actual[lo:hi+1]
            valid = np.isfinite(hist) & (truth != 0)
            scores.append(np.mean(np.sign(hist[valid]) == truth[valid]) if valid.sum() == selection_lookback else -1.0)
        best = int(np.argmax(scores))
        if scores[best] >= 0 and np.isfinite(pred[best, t]):
            out[t] = pred[best, t] * (-1 if invert else 1)
    return out


In [3]:
def make_dmd_scores(assets: pl.DataFrame, shard: int, shard_count: int) -> pl.DataFrame:
    """Make DMD scores for one group of stocks."""
    # Use the same trade dates for each stock.
    stock_ids = (assets.filter(pl.col('in_universe')).select('barrid').unique()
                 .sort('barrid')['barrid'].to_list()[shard::shard_count])
    source = assets.select('date', 'barrid', 'price', 'in_universe')
    calendar = source.select('date').unique().sort('date')
    rows = []
    for stock in source.filter(pl.col('barrid').is_in(stock_ids)).partition_by('barrid'):
        aligned = calendar.join(stock, on='date', how='left', validate='1:1').sort('date')
        # The model uses prices that are known on the signal date.
        scores = walk_forward_stock_signal(aligned['price'].to_numpy())
        rows.append(aligned.with_columns(pl.Series('dmd', scores)).filter(
            pl.col('in_universe') & (pl.col('date') >= __import__('datetime').date(2005, 1, 1))
        ).select(pl.col('date').alias('signal_date'), 'barrid', 'dmd'))
    return pl.concat(rows)


## 1.2 Make the inflation beta score

The code uses the CPI value that was available at each month end. It turns daily stock returns into monthly returns. It then measures how the stock return moves with the CPI change. The score is the negative of this value. Thus, a stock with more negative inflation sensitivity gets a higher score.

In [4]:
import polars as pl
def vintage_inflation_changes(vintages: pl.DataFrame) -> pl.DataFrame:
    """Calculate the last available month-over-month change in YoY CPI per vintage."""
    required = {"vintage_date", "observation_date", "cpi"}
    if missing := required - set(vintages.columns):
        raise ValueError(f"Missing CPI vintage fields: {sorted(missing)}")
    vintage_rows = (
        vintages.sort(["vintage_date", "observation_date"])
        .with_columns(
            ((pl.col("cpi") / pl.col("cpi").shift(12).over("vintage_date") - 1) * 100)
            .alias("inflation_yoy")
        )
        .with_columns(
            (pl.col("inflation_yoy") - pl.col("inflation_yoy").shift(1).over("vintage_date"))
            .alias("inflation_change")
        )
        .drop_nulls("inflation_change")
        .group_by("vintage_date", maintain_order=True)
        .last()
        .select(pl.col("vintage_date").alias("date"), "inflation_change")
        .sort("date")
    )
    return vintage_rows


def make_inflation_scores(assets: pl.DataFrame, vintages: pl.DataFrame) -> pl.DataFrame:
    """Make a dated stock score from inflation sensitivity."""
    # Use the CPI value that was available at each month end.
    cpi = vintage_inflation_changes(vintages)
    # Stock daily returns are percentage points. Convert them to decimals.
    monthly = (
        assets.lazy().select(pl.col('date').dt.month_end(), 'barrid', 'return')
        .group_by('date', 'barrid')
        .agg(((1 + pl.col('return') / 100).product() - 1).alias('stock_return'),
             pl.col('return').count().alias('n_returns'))
        .filter(pl.col('n_returns') > 0)
        .join(cpi.lazy(), on='date').sort('date').collect()
    )
    # Use up to 60 months. Require at least 36 valid months.
    result = (
        monthly.with_columns((pl.col('stock_return') * pl.col('inflation_change')).alias('xy'),
                             pl.col('inflation_change').pow(2).alias('yy'))
        .rolling('date', period='60mo', group_by='barrid')
        .agg(pl.len().alias('n'), pl.col('stock_return').mean().alias('x'),
             pl.col('inflation_change').mean().alias('y'),
             pl.col('xy').mean(), pl.col('yy').mean())
        .with_columns((pl.col('yy') - pl.col('y').pow(2)).alias('variance'))
        .filter((pl.col('n') >= 36) & (pl.col('variance') > 0))
        .select(pl.col('date').alias('available_date'), 'barrid',
                (-(pl.col('xy') - pl.col('x') * pl.col('y')) / pl.col('variance')).alias('inflation_beta'))
    )
    return result

def join_dated_scores(dmd_shards: list[pl.DataFrame], inflation: pl.DataFrame) -> pl.DataFrame:
    """Join each stock score to the date when it is known."""
    return (pl.concat(dmd_shards).sort('signal_date')
            .join_asof(inflation.sort('available_date'), left_on='signal_date',
                       right_on='available_date', by='barrid', strategy='backward', tolerance='45d')
            .select('signal_date', 'barrid', 'dmd', 'inflation_beta'))


### 1.2.1 Make the dated CPI source file

The Federal Reserve ALFRED service gives the CPI value that was available at each month end. The next code cell shows how to save those dated values. Keep that file if you need an exact later run.

In [5]:
import calendar
import csv
import io
from datetime import date
from urllib.request import urlopen
ALFRED_URL = 'https://alfred.stlouisfed.org/graph/alfredgraph.csv'
def fetch_alfred_month_end_vintages(start: date, end: date) -> pl.DataFrame:
    """Download one point-in-time CPIAUCSL snapshot per month from ALFRED.

    ``vintage_date`` is month end, not the observation month. Consumers must
    join the latest vintage whose date is on or before the signal date.
    """
    rows = []
    year, month = start.year, start.month
    while (year, month) <= (end.year, end.month):
        vintage = date(year, month, calendar.monthrange(year, month)[1])
        url = f"{ALFRED_URL}?id=CPIAUCSL&vintage_date={vintage.isoformat()}"
        with urlopen(url, timeout=30) as response:
            csv_text = response.read().decode("utf-8")
        for row in csv.DictReader(io.StringIO(csv_text)):
            value = row[f"CPIAUCSL_{vintage:%Y%m%d}"]
            if value and value != ".":
                if date.fromisoformat(row["observation_date"]) >= vintage.replace(day=1):
                    raise ValueError(f"Unexpected observation in vintage {vintage}: {row}")
                rows.append({
                    "vintage_date": vintage,
                    "observation_date": date.fromisoformat(row["observation_date"]),
                    "cpi": float(value),
                })
        if month == 12:
            year, month = year + 1, 1
        else:
            month += 1
    return pl.DataFrame(rows).sort(["vintage_date", "observation_date"])


# With network access, make the dated CPI source file once.
# cpi = fetch_alfred_month_end_vintages(date(2000, 1, 1), date(2025, 12, 31))
# cpi.write_parquet(Path('/path/to/licensed/source/cpi_alfred_vintages.parquet'))


## 1.3 Make the dated input files

Join each DMD score to the latest inflation score known at that time. The panel code aligns the signal date, trade date, and holding return. It also saves the benchmark weights. The risk files save daily stock risk in the same stock order as the panel. These files are in `dataset/`.

In [6]:
import polars as pl
def build_panel(assets: pl.DataFrame, forecasts: pl.DataFrame, exits: pl.DataFrame | None = None) -> pl.DataFrame:
    """Align dates on a common exchange calendar, including for stocks that exit."""
    calendar = assets.select("date").unique().sort("date").with_columns(
        pl.col("date").shift(1).alias("signal_date"),
        pl.col("date").shift(-1).alias("label_available_date"),
    )
    members = assets.filter(pl.col("in_universe")).with_columns(
        (pl.col("market_cap") / pl.col("market_cap").sum().over("date")).alias("benchmark_weight")
    )
    # sf-quant stores daily returns in percentage points. CRSP fallbacks are decimals.
    realized = assets.select(
        pl.col("date").alias("label_available_date"),
        "barrid",
        (pl.col("return") / 100).alias("holding_return"),
    )
    if exits is not None:
        realized = realized.join(
            exits.select("label_available_date", "barrid", "delret", "crsp_return"),
            on=["label_available_date", "barrid"], how="full", coalesce=True, validate="1:1",
        ).with_columns(
            pl.coalesce("holding_return", "delret", "crsp_return").alias("holding_return")
        ).drop("delret", "crsp_return")
    panel = (
        members.join(calendar, on="date", validate="m:1")
        .drop_nulls(["signal_date", "label_available_date"])
        .join(forecasts, on=["signal_date", "barrid"], how="left", validate="m:1")
        .join(realized, on=["label_available_date", "barrid"], how="left", validate="m:1")
        .with_columns(
            pl.col("holding_return").is_null().sum().over("date").alias("carried_names"),
            pl.col("holding_return").fill_null(0),
        )
        .with_columns(
            (pl.col("benchmark_weight") * pl.col("holding_return")).sum().over("date").alias("benchmark_return")
        )
        .with_columns(
            (pl.col("holding_return") - pl.col("predicted_beta") * pl.col("benchmark_return"))
            .alias("forward_residual_return")
        )
    )
    return panel.sort(["date", "barrid"])


In [7]:
def save_risk_year(panel: pl.DataFrame, risk_source, year: int, destination: Path) -> None:
    """Save the daily risk data for one calendar year."""
    arrays = {}
    days = panel.filter(pl.col('date').dt.year() == year).partition_by('date', maintain_order=True)
    for day in days:
        key = str(day['date'][0])
        ids = day['barrid'].to_list()
        # Save stock IDs and risk arrays in the same order.
        exposure, factor_risk, stock_risk = risk_source.construct_factor_model_components(day['date'][0], ids)
        arrays[key + '_ids'] = np.asarray(ids, dtype=str)
        arrays[key + '_E'] = exposure
        arrays[key + '_F'] = factor_risk
        arrays[key + '_S'] = stock_risk
    np.savez_compressed(destination, **arrays)


### 1.3.1 Add stock exit returns

A stock can leave the data after it is in a portfolio. The next code cell gets a return from the licensed CRSP files when the main stock file has no return. This step keeps the stock in the test for its last holding period.

In [8]:
def make_exit_audit(database: Path, destination: Path) -> None:
    """Find a return for a stock that leaves the stock data."""
    from datetime import date
    delist = pl.scan_parquet(database / 'production/crsp_delisting/crsp_delisting.parquet')
    rows = []
    for year in range(2005, 2026):
        paths = [database / 'production/assets' / f'assets_{y}.parquet'
                 for y in range(year, min(year + 1, 2025) + 1)]
        assets = pl.scan_parquet(paths)
        if year < 2025:
            assets = assets.filter(pl.col('date') <= date(year + 1, 1, 15))
        calendar = assets.select('date').unique().sort('date').with_columns(
            pl.col('date').shift(-1).alias('label_available_date'))
        # Select stocks with no return on the next trade date.
        missing = (
            assets.filter(pl.col('in_universe') & (pl.col('date').dt.year() == year))
            .select('date', 'barrid', pl.col('cusip').str.slice(0, 8).alias('cusip'))
            .join(calendar, on='date').drop_nulls('label_available_date')
            .join(assets.select(pl.col('date').alias('label_available_date'), 'barrid', 'return'),
                  on=['label_available_date', 'barrid'], how='left')
            .filter(pl.col('return').is_null()))
        crsp = pl.scan_parquet(database / 'research/crsp_daily' / f'crsp_daily_{year}.parquet')
        mapping = crsp.select('date', 'cusip', 'permno').drop_nulls().unique().group_by(
            'date', 'cusip').agg(pl.col('permno').first(), pl.len().alias('count')).filter(pl.col('count') == 1)
        return_paths = [database / 'research/crsp_daily' / f'crsp_daily_{y}.parquet'
                        for y in range(year, min(year + 1, 2025) + 1)]
        crsp_returns = pl.scan_parquet(return_paths).select(
            pl.col('date').alias('label_available_date'), 'permno',
            pl.when(pl.col('ret').is_finite() & (pl.col('ret') >= -1))
            .then(pl.col('ret')).alias('crsp_return'))
        matched = (
            missing.join(mapping, on=['date', 'cusip'], how='left')
            .join(delist.select('permno', pl.col('delistingdt').alias('label_available_date'), 'delret'),
                  on=['permno', 'label_available_date'], how='left', validate='m:1')
            .join(crsp_returns, on=['permno', 'label_available_date'], how='left', validate='m:1')
            .select('date', 'label_available_date', 'barrid', 'permno', 'delret', 'crsp_return').collect())
        rows.append(matched)
    pl.concat(rows).write_parquet(destination)

# Use the licensed CRSP files to make the source audit file.
# make_exit_audit(Path('/path/to/licensed/database'),
#                 Path('/path/to/licensed/source/exit_return_audit.parquet'))


In [9]:
def make_all_input_files(source_folder: Path, risk_source) -> None:
    """Make the dated panel and all yearly risk files from source data."""
    from datetime import date

    # Read the same stock fields for the full warm-up period.
    stock_files = [source_folder / f'assets_{year}.parquet' for year in range(1995, 2026)]
    assets = pl.scan_parquet(stock_files).select(
        'date', 'barrid', 'in_universe', 'market_cap', 'return',
        'predicted_beta', 'price').collect()
    vintages = pl.read_parquet(source_folder / 'cpi_alfred_vintages.parquet')
    exits = pl.read_parquet(source_folder / 'exit_return_audit.parquet')

    # Divide stocks into groups. Each group can run on a separate worker.
    # Keep the stock history and the full exchange calendar in each group.
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(max_workers=8) as pool:
        dmd_groups = list(pool.map(lambda i: make_dmd_scores(assets, i, 128), range(128)))
    inflation = make_inflation_scores(assets, vintages)
    forecasts = join_dated_scores(dmd_groups, inflation)

    # The portfolio test starts in 2010. Keep 2005 to 2009 for training.
    test_assets = assets.filter(pl.col('date').is_between(date(2005, 1, 1), END))
    raw_panel = build_panel(test_assets, forecasts, exits)

    # Add the two Barra style scores known on each signal date.
    parts = []
    years = raw_panel.with_columns(pl.col('signal_date').dt.year().alias('_year'))
    for year in years.partition_by('_year', include_key=False):
        styles = risk_source.load_exposures(
            year['signal_date'].min(), year['signal_date'].max(), False,
            ['USSLOWL_MOMENTUM', 'USSLOWL_LTREVRSL'])
        parts.append(year.join(styles.rename({'date': 'signal_date'}),
                               on=['signal_date', 'barrid'], how='left', validate='m:1'))
    panel = pl.concat(parts).sort('date', 'barrid')
    panel.write_parquet(DATA / 'analysis_panel.parquet')

    # The risk arrays must have the same stock order as the dated panel.
    for year in range(2010, 2025):
        save_risk_year(panel, risk_source, year, DATA / f'risk_{year}.npz')

# With licensed source files and sf-quant access, use:
# import sf_quant.data as risk_source
# risk_source.env(root=str(Path.home()), database='production')
# make_all_input_files(Path('/path/to/licensed/source'), risk_source)


In [10]:
# Check the supplied input files before the portfolio test.
import pyarrow.parquet as pq
panel_file = pq.ParquetFile(DATA / 'analysis_panel.parquet')
need = {'date', 'signal_date', 'label_available_date', 'barrid', 'dmd',
        'inflation_beta', 'holding_return', 'benchmark_weight',
        'USSLOWL_MOMENTUM', 'USSLOWL_LTREVRSL'}
assert need.issubset(panel_file.schema.names)
assert len(list(DATA.glob('risk_20*.npz'))) == 15
print(f"Stock rows: {panel_file.metadata.num_rows:,}")
print('Risk years: 15')


Stock rows: 14,939,812
Risk years: 15


## 1.4 Remove two style effects from DMD

For each date, the code fits DMD scores to Barra momentum and long-term reversal scores across stocks. It subtracts the fitted part from DMD. It does not change the inflation score. It also keeps the original DMD score for a matched test.

## 1.5 Fit three time-decay ridge rules

The code uses only holding returns known before each trade. It gives less weight to older results. The three half-lives are 63, 252, and 1,260 trade days. Their penalties are 18, 5, and 2.5. The penalty pulls both signal coefficients to the same value, 0.00001. Equal coefficients are the starting assumption. The code also tests a rule that gives the two standardized scores equal weight.

In [11]:
from dataclasses import dataclass
from datetime import date
from types import SimpleNamespace
SIGNALS = ('dmd', 'inflation_beta')
@dataclass(frozen=True)
class RidgeFit:
    as_of: date
    half_life_days: float
    penalty: float
    intercept: float
    coefficients: np.ndarray
    means: np.ndarray
    scales: np.ndarray
    n_observations: int

    def predict(self, signal_matrix: np.ndarray) -> np.ndarray:
        x = np.asarray(signal_matrix, dtype=float)
        if x.ndim != 2 or x.shape[1] != self.coefficients.size:
            raise ValueError(f"Expected matrix with {self.coefficients.size} signal columns")
        return self.intercept + ((x - self.means) / self.scales) @ self.coefficients


def daily_ridge_inputs(observations: pl.DataFrame, signals=SIGNALS):
    """Compress labeled rows to daily sufficient statistics for repeated fits."""
    target = "forward_residual_return"
    columns = (*signals, target)
    products = [(a, b) for i, a in enumerate(signals) for b in columns[i:]]
    daily = (
        observations.filter(pl.all_horizontal(pl.col(c).is_finite() for c in columns))
        .group_by("signal_date", "label_available_date")
        .agg(pl.len().alias("n"), *(pl.col(c).sum() for c in columns),
             *((pl.col(a) * pl.col(b)).sum().alias(f"{a}/{b}") for a, b in products))
        .sort("signal_date")
    )
    return daily, products, signals


def fit_daily_ridge(inputs, *, as_of: date, half_life_days: float, penalty: float, center: float = 0.0) -> RidgeFit:
    """Fit trailing standardized ridge from the same moments as the row-level fit."""
    daily, products, signals = inputs
    history = daily.filter(pl.col("label_available_date") < as_of)
    n = history["n"].to_numpy()
    if n.sum() < 30:
        raise ValueError(f"Only {n.sum()} matured observations before {as_of}")
    weights = np.exp2(-np.arange(len(n) - 1, -1, -1) / half_life_days)
    total = weights @ n
    means = weights @ history.select(*signals, "forward_residual_return").to_numpy() / total
    weighted_products = weights @ history.select(*(f"{a}/{b}" for a, b in products)).to_numpy() / total
    covariance = np.zeros((len(signals), len(signals)))
    cross = np.zeros(len(signals))
    for (a, b), moment in zip(products, weighted_products):
        i = signals.index(a)
        if b in signals:
            j = signals.index(b)
            covariance[i, j] = covariance[j, i] = moment - means[i] * means[j]
        else:
            cross[i] = moment - means[i] * means[-1]
    scales = np.maximum(np.sqrt(np.maximum(covariance.diagonal(), 0)), 1e-12)
    gram = covariance / scales[:, None] / scales[None, :]
    coefficients = np.linalg.solve(gram + penalty * np.eye(len(signals)), cross / scales + penalty * center)
    return RidgeFit(as_of, half_life_days, penalty, means[-1], coefficients,
                    means[:-1], scales, int(n.sum()))


# Keep the two fit functions together for the backtest.
combination = SimpleNamespace(daily_ridge_inputs=daily_ridge_inputs,
                              fit_daily_ridge=fit_daily_ridge)


## 1.6 Find stock weights

Each total portfolio has a weight sum of one. Its active market beta is zero. Its forecast tracking error is at most 1% per year. The long-only portfolio has no negative stock weights. The other portfolio can have negative stock weights.

In [12]:
import clarabel
from scipy import sparse
def solve(
    alpha, exposures, covariance, specific_variance, benchmark, beta,
    *, long_only, risk_cap,
):
    """Maximize alpha under an exact second-order cone tracking-error cap.

    Decision variables are active weights and their factor exposures. Keeping
    factor exposures explicit preserves sparsity in the stock risk block.
    """
    stocks, factors = exposures.shape
    size = stocks + factors
    eig, vectors = np.linalg.eigh(covariance)
    if eig.min() < -1e-10:
        raise ValueError("Barra factor covariance is not positive semidefinite")
    root = vectors * np.sqrt(np.maximum(eig, 0))
    equalities = np.vstack([np.ones(stocks), beta])
    a = [
        sparse.bmat([
            [sparse.csc_matrix(equalities), sparse.csc_matrix((2, factors))],
            [-sparse.csc_matrix(exposures.T), sparse.eye(factors)],
        ]),
        sparse.csc_matrix((1, size)),
        sparse.bmat([
            [sparse.csc_matrix((factors, stocks)), -sparse.csc_matrix(root.T)],
            [-sparse.diags(np.sqrt(specific_variance)), sparse.csc_matrix((stocks, factors))],
        ]),
    ]
    b = [np.zeros(2 + factors), np.r_[risk_cap, np.zeros(size)]]
    cones = [clarabel.ZeroConeT(2 + factors), clarabel.SecondOrderConeT(1 + size)]
    if long_only:
        a.append(sparse.hstack([-sparse.eye(stocks), sparse.csc_matrix((stocks, factors))]))
        b.append(benchmark)
        cones.append(clarabel.NonnegativeConeT(stocks))
    settings = clarabel.DefaultSettings()
    settings.verbose = False
    settings.max_threads = 1
    result = clarabel.DefaultSolver(
        sparse.csc_matrix((size, size)), np.r_[-alpha, np.zeros(factors)],
        sparse.vstack(a, format="csc"), np.concatenate(b), cones, settings,
    ).solve()
    if result.status not in (clarabel.SolverStatus.Solved, clarabel.SolverStatus.AlmostSolved):
        raise ValueError(f"Sparse Barra optimizer failed: {result.status}")
    weights = np.asarray(result.x[:stocks])
    risk = np.linalg.norm(np.r_[root.T @ (exposures.T @ weights), np.sqrt(specific_variance) * weights])
    return weights, float(risk)


sparse_optimizer = SimpleNamespace(solve=solve)


## 1.7 Set the test values

In [13]:
PENALTIES = {63: 18, 252: 5, 1260: 2.5}
FIXED_PRIOR = 1e-5
TE_CAP = 0.01
END = date(2024, 12, 31)
SIGNALS = ('dmd', 'inflation_beta')


## 1.8 Run the stock tests and update model weights

The next function does the full test. It first makes the adjusted DMD score. It then fits the three ridge rules on each date. It tests these rules and the equal score rule as active portfolios. It also tests each signal alone. After a holding return is known, it uses that active return to update the Bayesian model weights. It uses the same model weights for the long-only combined portfolio. It saves returns, coefficients, model weights, factor exposures, and style data in `dataset/`. The code uses several date workers to reduce run time.

In [14]:
def run_final(baselines_only=False):
    if not baselines_only:
        assert all(v is not None and v>=0 for v in PENALTIES.values()), 'Set final penalties first'
    # Load scores, returns, dates, and stock IDs.
    panel=pl.read_parquet(DATA/'analysis_panel.parquet').sort('date','barrid')
    diagnostics=[];parts=[]
    # Remove the Barra momentum and reversal parts from DMD each day.
    for day in panel.partition_by('date',maintain_order=True):
        raw=day['dmd'].to_numpy(); styles=day.select('USSLOWL_MOMENTUM','USSLOWL_LTREVRSL').to_numpy()
        mask=np.isfinite(raw)&np.isfinite(styles).all(axis=1)
        if mask.sum()<30: raise ValueError('Insufficient style coverage')
        x=styles[mask]-styles[mask].mean(axis=0); y=raw[mask]-raw[mask].mean()
        residual=y-x@np.linalg.lstsq(x,y,rcond=None)[0]
        clean=np.zeros(len(raw));matched=np.zeros(len(raw));clean[mask]=residual;matched[mask]=y
        diagnostics.append({'date':day['date'][0],'coverage':mask.mean(),'r2':1-residual@residual/(y@y) if y@y else np.nan})
        parts.append(day.with_columns(pl.Series('dmd',clean),pl.Series('raw_dmd',matched)))
    panel=pl.concat(parts)
    pl.DataFrame(diagnostics).write_parquet(DATA/'style_diagnostics.parquet')
    # Build daily training sums from known holding returns.
    inputs=combination.daily_ridge_inputs(panel,signals=SIGNALS)
    days=panel.filter((pl.col('date').dt.year()>=2010)&(pl.col('label_available_date')<=END)).partition_by('date',maintain_order=True)
    dates=[d['date'][0] for d in days]
    fits={} if baselines_only else {dt:{f'ridge_{half}_p{penalty:g}':(combination.fit_daily_ridge(inputs,as_of=dt,half_life_days=half,penalty=penalty,center=FIXED_PRIOR),FIXED_PRIOR) for half,penalty in PENALTIES.items()} for dt in dates}
    import threading, os
    from concurrent.futures import ThreadPoolExecutor
    rows=[];coefs=[];factor_rows=[];worker_state=threading.local()
    workers=min(8,int(os.environ.get('SLURM_CPUS_PER_TASK','4')))
    # Find stock weights under the risk and beta limits.
    def evaluate(day, name, alpha, long_only):
        dt=day['date'][0];key=str(dt)
        if getattr(worker_state,'risk_year',None)!=dt.year:
            if getattr(worker_state,'archive',None) is not None: worker_state.archive.close()
            worker_state.archive=np.load(DATA/f'risk_{dt.year}.npz',allow_pickle=False)
            worker_state.risk_year=dt.year
        archive=worker_state.archive
        assert list(archive[key+'_ids'])==day['barrid'].to_list()
        E,F,S=[archive[key+'_'+s] for s in ('E','F','S')]
        b=day['benchmark_weight'].to_numpy();beta=day['predicted_beta'].to_numpy()
        a,te=sparse_optimizer.solve(alpha/max(np.max(np.abs(alpha)),1e-20),E,F,S,b,beta,long_only=long_only,risk_cap=TE_CAP)
        total=b+a; ret=day['holding_return'].to_numpy()
        assert abs(total.sum()-1)<1e-5 and abs(a@beta)<1e-5 and te<=TE_CAP+1e-6
        if long_only: assert total.min()>-1e-6
        variant='long_only' if long_only else 'active';label=name+'_'+variant
        exposures=E.T@a
        factor_rows.extend({'date':dt,'portfolio':label,'factor_index':j,'active_exposure':float(v),'variance_contribution':float(v*(F@exposures)[j])} for j,v in enumerate(exposures))
        return dict(date=day['label_available_date'][0],portfolio=label,active_return=float(a@ret),portfolio_return=float(total@ret),benchmark_return=float(b@ret),predicted_te=te,active_beta=float(a@beta),total_investment=float(total.sum()),minimum_total_weight=float(total.min()),specific_variance=float((a*a)@S),lagged_momentum_exposure=float(a@np.nan_to_num(day['USSLOWL_MOMENTUM'].to_numpy())),lagged_reversal_exposure=float(a@np.nan_to_num(day['USSLOWL_LTREVRSL'].to_numpy())))
    def alphas(day):
        dt=day['date'][0];scores=day.select(SIGNALS).to_numpy();out={}
        for name,(fit,center) in fits.get(dt,{}).items():
            z=np.nan_to_num((scores-fit.means)/fit.scales)
            out[name]=z@fit.coefficients
        f=combination.fit_daily_ridge(inputs,as_of=dt,half_life_days=126,penalty=1)
        z=np.nan_to_num((scores-f.means)/f.scales)
        out['equal_weight']=z.mean(axis=1)
        return out
    # Test the three ridge rules and the equal score rule.
    def candidate_day(day):
        dt=day['date'][0];candidates=alphas(day)
        coefficients=[dict(date=dt,method=name,center=center,**dict(zip(SIGNALS,fit.coefficients))) for name,(fit,center) in fits.get(dt,{}).items()]
        candidates.update(dmd_standalone=day['dmd'].to_numpy(),inflation_standalone=np.nan_to_num(day['inflation_beta'].to_numpy()),original_dmd_matched=day['raw_dmd'].to_numpy())
        result=[]
        for name,alpha in candidates.items():
            variants=(False,True) if name in ('dmd_standalone','inflation_standalone','original_dmd_matched') else (False,)
            for lo in variants: result.append(evaluate(day,name,alpha,lo))
        return result,coefficients
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for index,(result,coefficients) in enumerate(pool.map(candidate_day,days),1):
            rows.extend(result);coefs.extend(coefficients)
            if index%250==0: print('Backtest',index,'/',len(days),flush=True)
    history=pl.DataFrame(rows)
    if baselines_only:
        history.write_parquet(DATA/'baseline_returns.parquet')
        pl.DataFrame(factor_rows).write_parquet(DATA/'baseline_factor_exposures.parquet')
        print('Penalty-independent backtests complete',flush=True)
        return
    names=list(fits[dates[0]])+['equal_weight']
    from scipy.special import softmax
    columns=[name+'_active' for name in names]
    active_history=history.filter(pl.col('portfolio').is_in(columns)).pivot(on='portfolio',index='date',values='active_return').sort('date')
    assert set(active_history.columns)=={'date',*columns} and active_history.height==len(dates)
    realized=active_history.select(columns).to_numpy()
    assert np.isfinite(realized).all()
    known=np.searchsorted(active_history['date'].to_numpy(),np.asarray(dates,dtype='datetime64[D]'),side='left')
    cumulative=np.vstack([np.zeros(len(names)),realized.cumsum(axis=0)])
    # Update model weights with active returns that are known now.
    active_post=dict(zip(dates,softmax((0.01/0.025**2)*cumulative[known],axis=1)))
    post={'active':active_post,'long_only':active_post}
    # Use these model weights for both portfolio types.
    def bayes_day(day):
        dt=day['date'][0];a=alphas(day);matrix=np.column_stack([a[n] for n in names]);sd=matrix.std(axis=0)
        matrix=np.divide(matrix,sd,out=np.zeros_like(matrix),where=sd>0)
        return [evaluate(day,'bayesian',matrix@post['long_only' if lo else 'active'][dt],lo) for lo in (False,True)]
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for index,result in enumerate(pool.map(bayes_day,days),1):
            rows.extend(result)
            if index%250==0: print('Bayesian backtest',index,'/',len(days),flush=True)
    # Save all results for the two analysis notebooks.
    pl.DataFrame(rows).write_parquet(DATA/'final_returns.parquet')
    pl.DataFrame(coefs).write_parquet(DATA/'final_coefficients.parquet')
    pl.DataFrame(factor_rows).write_parquet(DATA/'final_factor_exposures.parquet')
    pl.DataFrame([dict(date=dt,variant=v,**dict(zip(names,w))) for v,series in post.items() for dt,w in series.items()]).write_parquet(DATA/'final_model_weights.parquet')
# run_final() after the dataset export.


## 1.9 Start the full test

Run the next cell on a computer with at least 16 GB of memory and eight CPU cores. The full test can take more than 15 minutes. The supplied result files already contain its output. Do not start this cell if you need to keep the 15-minute run limit.

In [15]:
# Remove the comment mark on the next line to make the full result files.
# run_final()


## 1.10 Check the result files

The reference files give the result for each date and portfolio. After a new run, compare the result files with them. A very small number difference can come from the math solver. The check below stops if the difference is larger than 0.0000001.

In [16]:
# Check the dated rows and the main numeric results.
checks = [
    ('final_returns.parquet', 'reference_returns.parquet', ['date', 'portfolio'],
     ['active_return', 'portfolio_return', 'benchmark_return', 'predicted_te', 'active_beta']),
    ('final_coefficients.parquet', 'reference_coefficients.parquet', ['date', 'method'],
     ['dmd', 'inflation_beta', 'center']),
    ('final_model_weights.parquet', 'reference_model_weights.parquet', ['date', 'variant'],
     ['ridge_63_p18', 'ridge_252_p5', 'ridge_1260_p2.5', 'equal_weight'])]
for current_name, reference_name, keys, fields in checks:
    current = pd.read_parquet(DATA / current_name).sort_values(keys).reset_index(drop=True)
    reference = pd.read_parquet(DATA / reference_name).sort_values(keys).reset_index(drop=True)
    assert current[keys].equals(reference[keys])
    for field in fields:
        difference = np.nanmax(np.abs(current[field].to_numpy() - reference[field].to_numpy()))
        assert difference < 1e-7, (current_name, field, difference)
    print(f'{current_name}: {len(current):,} checked rows')


final_returns.parquet: 45,276 checked rows
final_coefficients.parquet: 11,319 checked rows
final_model_weights.parquet: 7,546 checked rows
